## Homework 6 

In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score, root_mean_squared_error
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

In [2]:
df_raw = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')
df_raw.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [13]:
def clean_and_normalize_raw_data(df: pd.DataFrame):
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)
    numerical_columns = list(df.dtypes[df.dtypes != 'str'].index)

    for column in categorical_columns:
        df[column] = df[column].str.lower().replace(' ', '_')
        df[column] = df[column].fillna("NA")

    for column in numerical_columns:
        df[column] = df[column].fillna(0.0)

    return df

In [14]:
df_normalized = clean_and_normalize_raw_data(df_raw)
df_normalized.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,europe,gasoline,front-wheel drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,europe,diesel,front-wheel drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,asia,gasoline,front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,europe,gasoline,front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,usa,diesel,front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [19]:
def validation_framework_generator(data: pd.DataFrame, f_size: float, s_size:float, r_state: int, obj_variable: str):
    df_full_train, df_test = train_test_split(data, test_size=f_size, random_state=r_state)
    df_train, df_val = train_test_split(df_full_train, test_size=s_size, random_state=r_state)

    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)
    df_full_train = df_full_train.reset_index(drop=True)

    y_train = df_train[obj_variable].values
    y_val = df_val[obj_variable].values
    y_test = df_test[obj_variable].values
    y_full_train = df_full_train[obj_variable].values

    del df_train[obj_variable]
    del df_val[obj_variable]
    del df_test[obj_variable]
    del df_full_train[obj_variable]

    return df_train, y_train, df_val, y_val, df_test, y_test, df_full_train, y_full_train
    

In [21]:
df_train, y_train, df_val, y_val, df_test, y_test, df_full_train, y_full_train = validation_framework_generator(data=df_normalized,
                                                                                                               f_size=0.2,
                                                                                                               s_size=0.25,
                                                                                                               r_state=1,
                                                                                                               obj_variable='fuel_efficiency_mpg')

In [25]:
dv = DictVectorizer(sparse=False)

In [29]:
def to_vetorized_matrix(data: pd.DataFrame, fit: bool = False):
    dicts = data.to_dict(orient='records')

    if fit:
        return dv.fit_transform(dicts)
        
    return dv.transform(dicts)

In [34]:
def decision_tree_training(X_train, y_train, **kwargs):
    dt = DecisionTreeRegressor(**kwargs)
    dt.fit(X_train, y_train)

    return dt

In [30]:
X_train = to_vetorized_matrix(df_train, fit=True)

In [42]:
X_val = to_vetorized_matrix(df_val)

In [35]:
dt1 = decision_tree_training(X_train, y_train, max_depth=1)

In [37]:
print(export_text(dt1, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [39]:
def random_forest_training(X_train, y_train, **kwargs):
    rf = RandomForestRegressor(**kwargs)
    rf.fit(X_train, y_train)
    return rf

In [54]:
def rmse_evaluator(**kwargs):
    rfe = random_forest_training(X_train, y_train, **kwargs)

    y_prede = rfe.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_prede)

    return rmse

In [40]:
rf1 = random_forest_training(X_train, y_train, n_estimators=10, random_state=1, n_jobs=-1)

In [45]:
y_pred = rf1.predict(X_val)

In [50]:
print(root_mean_squared_error(y_val, y_pred))

1.8336350509302555


In [55]:
estimators = [10, 50, 100, 150]
results = {}

for n in estimators:
    results[n] = rmse_evaluator(n_estimators=n, random_state=1)

print(results)

{10: 1.8336350509302553, 50: 1.7696263933384353, 100: 1.7675167068517343, 150: 1.767526827582038}


In [57]:
depths = [10, 15, 20, 25]
estimators = [10, 50, 100, 150]

n_results = {}

for d in depths:
    rmse_values = []
    for e in estimators:
        score = rmse_evaluator(n_estimators=e, max_depth=d, random_state=1)
        rmse_values.append(score)
    n_results[d] = np.mean(rmse_values)
    
print(n_results)

{10: np.float64(1.7568661294868555), 15: np.float64(1.78478180194174), 20: np.float64(1.78267768236131), 25: np.float64(1.787372763495381)}


In [60]:
rf2 = random_forest_training(X_train, y_train, n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)

In [61]:
importances = pd.Series(rf2.feature_importances_, index=dv.get_feature_names_out())

print(importances.sort_values(ascending=False))

model_year                      0.295822
vehicle_weight                  0.203649
fuel_type=gasoline              0.160713
horsepower                      0.078254
engine_displacement             0.061008
acceleration                    0.055466
origin=usa                      0.042232
fuel_type=diesel                0.021128
drivetrain=front-wheel drive    0.020005
num_doors                       0.019724
fuel_type=hybrid                0.011685
drivetrain=all-wheel drive      0.007873
num_cylinders                   0.006939
origin=asia                     0.006479
drivetrain=rear-wheel drive     0.005636
origin=europe                   0.003389
dtype: float64


In [64]:
def to_xgboost_matrix(data, label):
    features = list(dv.get_feature_names_out())
    return xgb.DMatrix(data, label=label, feature_names=features)

In [67]:
def train_xgboost_model(xgb_params, d_train, **kwargs):
    model = xgb.train(xgb_params, dtrain, **kwargs)
    return model

In [66]:
dtrain = to_xgboost_matrix(X_train, y_train)
dval = to_xgboost_matrix(X_val, y_val)

In [68]:
watchlist = [(dtrain, "train"),(dval, "val")]
etas = [0.3, 0.1]
results = {}

for eta in etas:
    xgb_params = {
        "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "nthread": 8,
        "seed": 1,
        "verbosity": 1
    }

    model = train_xgboost_model(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False
    )

    predictions = model.predict(dval)
    results[eta] = root_mean_squared_error(y_val, predictions)

print(results)

{0.3: 1.8264579799904583, 0.1: 1.724810059827976}
